# 00_env_config — shared FabricOps runtime bootstrap

This notebook is loaded by downstream FabricOps notebooks.
It initializes shared runtime configuration, helper functions, AI prompts, and fail-fast validations.

Typical usage in downstream notebooks:

```python
%run 00_env_config
```


In [ ]:
# Runtime parameters (edit placeholders for your workspace)
ENV = "dev"
AGREEMENT_ID = "replace_me"
SOURCE_LAYER = "source"
TARGET_LAYER = "unified"
AI_ENABLED = True
VALIDATION_MODE = "strict"  # strict or warn


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
import re
import uuid
from typing import Any

# Optional Fabric runtime handles are resolved lazily in helper functions.
try:
    spark  # type: ignore[name-defined]
except NameError:
    spark = None


In [ ]:
@dataclass(frozen=True)
class Housepath:
    """Represents one target location in a Fabric environment."""

    workspace_id: str
    item_id: str
    item_name: str
    abfss_path: str


@dataclass(frozen=True)
class RuntimeContext:
    """Shared runtime values used by downstream notebooks."""

    env: str
    agreement_id: str
    source_layer: str
    target_layer: str
    run_id: str
    run_timestamp: str
    ai_enabled: bool
    validation_mode: str


In [ ]:
# Placeholder environment registry (public-safe values only)
ENV_REGISTRY: dict[str, dict[str, Housepath]] = {
    "dev": {
        "source": Housepath("00000000-0000-0000-0000-000000000001", "11111111-1111-1111-1111-111111111111", "lh_source_dev", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_source_dev"),
        "unified": Housepath("00000000-0000-0000-0000-000000000001", "22222222-2222-2222-2222-222222222222", "lh_unified_dev", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_unified_dev"),
        "product": Housepath("00000000-0000-0000-0000-000000000001", "33333333-3333-3333-3333-333333333333", "lh_product_dev", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_product_dev"),
        "metadata": Housepath("00000000-0000-0000-0000-000000000001", "44444444-4444-4444-4444-444444444444", "lh_metadata_dev", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_metadata_dev"),
        "warehouse": Housepath("00000000-0000-0000-0000-000000000001", "55555555-5555-5555-5555-555555555555", "wh_ops_dev", "sql://warehouse/dev"),
    },
    "prod": {
        "source": Housepath("99999999-9999-9999-9999-999999999991", "aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa", "lh_source_prod", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_source_prod"),
        "unified": Housepath("99999999-9999-9999-9999-999999999991", "bbbbbbbb-bbbb-bbbb-bbbb-bbbbbbbbbbbb", "lh_unified_prod", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_unified_prod"),
        "product": Housepath("99999999-9999-9999-9999-999999999991", "cccccccc-cccc-cccc-cccc-cccccccccccc", "lh_product_prod", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_product_prod"),
        "metadata": Housepath("99999999-9999-9999-9999-999999999991", "dddddddd-dddd-dddd-dddd-dddddddddddd", "lh_metadata_prod", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_metadata_prod"),
        "warehouse": Housepath("99999999-9999-9999-9999-999999999991", "eeeeeeee-eeee-eeee-eeee-eeeeeeeeeeee", "wh_ops_prod", "sql://warehouse/prod"),
    },
}

TARGET_REGISTRY = ("source", "unified", "product", "metadata", "warehouse")


In [ ]:
def validate_environment(env: str) -> None:
    if env not in ENV_REGISTRY:
        raise ValueError(f"Unknown env '{env}'. Available: {sorted(ENV_REGISTRY)}")


def validate_target(target: str, env: str = ENV) -> None:
    validate_environment(env)
    available = ENV_REGISTRY[env].keys()
    if target not in available:
        raise ValueError(f"Unknown target '{target}' for env '{env}'. Available: {sorted(available)}")


def get_path(target: str, env: str = ENV) -> Housepath:
    validate_target(target, env)
    return ENV_REGISTRY[env][target]


def _validate_no_cross_environment_leakage() -> None:
    for env_name, targets in ENV_REGISTRY.items():
        for target_name, hp in targets.items():
            if env_name not in hp.item_name:
                raise ValueError(
                    f"Target '{target_name}' in env '{env_name}' should include env marker in item_name: {hp.item_name}"
                )


In [ ]:
def _resolve_spark():
    if spark is None:
        raise RuntimeError("Spark session is not available. Run in Fabric runtime for Spark operations.")
    return spark


def lakehouse_table_read(target: str, table_name: str, env: str = ENV):
    _ = get_path(target, env)
    return _resolve_spark().table(table_name)


def lakehouse_table_write(df, target: str, table_name: str, mode: str = "overwrite", env: str = ENV) -> None:
    _ = get_path(target, env)
    df.write.mode(mode).saveAsTable(table_name)


def lakehouse_csv_read(target: str, relative_path: str, env: str = ENV, header: bool = True):
    hp = get_path(target, env)
    return _resolve_spark().read.option("header", str(header).lower()).csv(f"{hp.abfss_path}/{relative_path}")


def warehouse_read(query: str, env: str = ENV):
    get_path("warehouse", env)
    raise NotImplementedError("Bind warehouse connector in your Fabric runtime before using warehouse_read.")


def warehouse_write(df, table_name: str, mode: str = "append", env: str = ENV) -> None:
    get_path("warehouse", env)
    raise NotImplementedError("Bind warehouse connector in your Fabric runtime before using warehouse_write.")


In [ ]:
METADATA_TABLES = {
    "profile_rows": "metadata.profile_rows",
    "dq_rules": "metadata.dq_rules",
    "lineage_events": "metadata.lineage_events",
    "handover_summaries": "metadata.handover_summaries",
}

OUTPUT_NAMING = {
    "unified_table_prefix": "unified_",
    "product_table_prefix": "product_",
}

TECHNICAL_COLUMNS = ["run_id", "run_timestamp", "source_system", "record_hash"]
AUDIT_COLUMNS = ["created_at", "created_by", "updated_at", "updated_by"]

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()


In [ ]:
def clean_datetime_columns(df, columns: list[str]):
    from pyspark.sql import functions as F

    for col in columns:
        df = df.withColumn(col, F.to_timestamp(F.col(col)))
    return df


def add_system_technical_columns(df, run_id: str = RUN_ID, run_timestamp: str = RUN_TIMESTAMP):
    from pyspark.sql import functions as F

    return (
        df.withColumn("run_id", F.lit(run_id))
        .withColumn("run_timestamp", F.to_timestamp(F.lit(run_timestamp)))
    )


def log_transformation_summary(step_name: str, row_count: int, details: str = "") -> dict[str, Any]:
    return {"step": step_name, "row_count": row_count, "details": details, "run_id": RUN_ID, "run_timestamp": RUN_TIMESTAMP}


def log_metadata_event(event_type: str, details: dict[str, Any]) -> dict[str, Any]:
    return {"event_type": event_type, "details": details, "run_id": RUN_ID, "run_timestamp": RUN_TIMESTAMP}


In [ ]:
AI_PROMPTS = {
    "business_context": "Summarize business context, approved usage, and policy boundaries for this dataset.",
    "dq_rule_suggestion": "Suggest candidate data quality rules from schema, profiling, and sample evidence.",
    "dq_rule_review": "Review candidate DQ rules for enforceability, clarity, and governance alignment.",
    "governance_classification": "Propose governance and sensitivity classification with rationale.",
    "profile_summary": "Summarize profiling findings, key anomalies, and follow-up checks.",
    "lineage_summary": "Summarize lineage from source through unified/product outputs.",
    "handover_summary": "Generate an implementation handover summary from approved metadata and runtime logs.",
}


In [ ]:
ALLOWED_NOTEBOOK_PREFIXES = (
    "00_env_config",
    "01_dsa_",
    "02_ex_",
    "03_pc_",
)


def check_naming_convention(notebook_name: str | None = None) -> bool:
    if notebook_name is None:
        notebook_name = "00_env_config"
    ok = any(notebook_name.startswith(prefix) for prefix in ALLOWED_NOTEBOOK_PREFIXES)
    if ok:
        return True
    message = f"Notebook name '{notebook_name}' does not match allowed prefixes: {ALLOWED_NOTEBOOK_PREFIXES}"
    if VALIDATION_MODE == "strict":
        raise ValueError(message)
    print(f"WARNING: {message}")
    return False


In [ ]:
def validate_runtime() -> None:
    # Required imports are already resolved in this notebook; check optional runtime markers.
    if VALIDATION_MODE == "strict" and spark is None:
        raise RuntimeError("Spark runtime not detected. Use Microsoft Fabric runtime for downstream notebooks.")


def validate_config() -> None:
    validate_environment(ENV)
    for target in ("source", "unified", "product", "metadata"):
        validate_target(target, ENV)
    if not AGREEMENT_ID or AGREEMENT_ID == "replace_me":
        message = "AGREEMENT_ID is still a placeholder."
        if VALIDATION_MODE == "strict":
            raise ValueError(message)
        print(f"WARNING: {message}")
    if not METADATA_TABLES:
        raise ValueError("METADATA_TABLES must be configured.")


def validate_paths() -> None:
    _validate_no_cross_environment_leakage()


def validate_ai_ready() -> None:
    if AI_ENABLED and "dq_rule_suggestion" not in AI_PROMPTS:
        raise ValueError("AI_PROMPTS must include required keys when AI_ENABLED=True")


def initialize_fabricops_runtime(notebook_name: str = "00_env_config") -> RuntimeContext:
    check_naming_convention(notebook_name)
    validate_runtime()
    validate_config()
    validate_paths()
    validate_ai_ready()
    return RuntimeContext(
        env=ENV,
        agreement_id=AGREEMENT_ID,
        source_layer=SOURCE_LAYER,
        target_layer=TARGET_LAYER,
        run_id=RUN_ID,
        run_timestamp=RUN_TIMESTAMP,
        ai_enabled=AI_ENABLED,
        validation_mode=VALIDATION_MODE,
    )


In [ ]:
RUNTIME_CONTEXT = initialize_fabricops_runtime()

print("FabricOps runtime loaded")
print(f"- env: {RUNTIME_CONTEXT.env}")
print(f"- agreement_id: {RUNTIME_CONTEXT.agreement_id}")
print(f"- source target: {get_path(SOURCE_LAYER).item_name}")
print(f"- target layer: {RUNTIME_CONTEXT.target_layer}")
print(f"- metadata target: {get_path('metadata').item_name}")
print(f"- AI enabled: {RUNTIME_CONTEXT.ai_enabled}")
print(f"- validation mode: {RUNTIME_CONTEXT.validation_mode}")
